# Colab: Hugging Face Pipelines (explained deeply)

Lives in **`7_HUGGING_FACE/`**. Upload / open in **Google Colab** to run (GPU helps).

**How this notebook is organized:** each section shows the **code cell first**, then the **explanation** under it.

**Habit:** new pipeline task examples (sentiment, QA, NER, …) go in **this notebook** — code first, explanation after.

| Notebook | Where the model runs | What you use |
|----------|----------------------|--------------|
| `8_GOOGLE_COLAB/colab_openai_and_hf.ipynb` | Remote API | chat call + keys |
| **This notebook** | Inside Colab | `transformers.pipeline` |

### Why Colab?

- Pipelines download model weights onto the machine
- Free **T4 GPU** helps (Runtime → Change runtime type → T4 GPU)

### Optional Secret

Many public models work without a token. If needed: Colab Secret `HF_TOKEN` + notebook access.

Also see: [`what_is_huggingface.md`](./what_is_huggingface.md) (Pipeline section).

Run **top → bottom**.

## Big picture — 3 ways to get an answer

```text
Tokenizer + Model  →  Pipeline  →  Remote API
(you wire steps)     (local easy)  (your HF/OpenAI scripts)
```

| Layer | What you do | Weights on this machine? |
|-------|-------------|--------------------------|
| Tokenizer + Model | Call each yourself | Yes |
| **Pipeline** | `pipeline("task")` then run text | Yes |
| Remote API | `chat.completions.create(...)` | No |

## Setup

In [ ]:
!pip -q install transformers accelerate sentencepiece

### Explanation — install

| Piece | Meaning |
|-------|---------|
| `!` | Shell command in Colab |
| `transformers` | Library that provides `pipeline` |
| `accelerate` | Helps use GPU |
| `sentencepiece` | Some tokenizers need this |
| `-q` | Quiet install |

New Colab runtime → install again.

In [ ]:
import torch

try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("HF_TOKEN: logged in")
    else:
        print("HF_TOKEN: not set (OK for many public models)")
except Exception as e:
    print("Secrets / login skipped:", e)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

### Explanation — token + GPU check

| Piece | Meaning |
|-------|---------|
| `userdata.get("HF_TOKEN")` | Read Colab Secret (if set) |
| `login(token=...)` | Hub login (gated models) |
| `torch.cuda.is_available()` | Is there a GPU? |

## Example 1 — Sentiment analysis

First run may download a default model (normal).

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
result = classifier("I love learning with Hugging Face pipelines!")
print(result)

print("\nAnother example:")
print(classifier("This movie was terrible and boring."))

### Explanation — the three lines

#### `from transformers import pipeline`

Only an **import**. Brings the helper function named `pipeline` into the notebook. **No model loaded yet.**

#### `classifier = pipeline("sentiment-analysis")`

**Build the tool.**

- `pipeline` = the **function**
- `"sentiment-analysis"` = **just a string** (task name), **not** another function
- Hugging Face looks up that string and loads a default tokenizer + model
- `classifier` = the ready **object/tool**

You are naming the **job**, not passing a function.

#### `result = classifier("I love...")`

**Use the tool.** Feed one sentence (another string). Inside: text → tokenizer → model → label.

```text
import pipeline          → get the factory
pipeline("sentiment...") → build tool → classifier
classifier("I love...")  → run sentence → result
```

More detail: [`what_is_huggingface.md`](./what_is_huggingface.md) → Pipeline → *Is `"sentiment-analysis"` another function, or just a string?*

## Example 2 — Same task, you choose the model

In [ ]:
classifier2 = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(classifier2("Pipelines make local inference much easier."))

### Explanation — explicit model

| Piece | Meaning |
|-------|---------|
| `"sentiment-analysis"` | Task name (**string**) |
| `model="distilbert/..."` | Hub model id (**also a string**) |
| Still `pipeline(...)` | Tokenizer + postprocess still done for you |

Better for real projects than relying on the silent default.

#### What is a classifier?

A **classifier** is a model whose job is to put the input into a **category / label** — not to write a long free answer.

```text
Input text  →  Classifier  →  Label (one of a fixed set)
"I love this!"              →  POSITIVE
"This was awful."           →  NEGATIVE
```

| | Classifier | Chat / generative model (e.g. Qwen Instruct) |
|--|------------|-----------------------------------------------|
| Output | A **label** from a fixed list | New **text** (sentences, answers) |
| Example | POSITIVE / NEGATIVE / SPAM / NOT_SPAM | “The capital of France is Paris.” |
| This notebook | DistilBERT sentiment | — |
| Your API scripts | — | Qwen chat |

So **“sentiment classifier”** means: a model trained to **classify** text by sentiment (pick a label), not chat with you.

Two uses of the word here:

| Word | Meaning |
|------|---------|
| `classifier = pipeline(...)` | Just a **variable name** for the pipeline tool |
| “classifier model” | The **kind of model** (label-picker), vs a chat LLM |

#### Important: `pipeline` is NOT the classifier itself

This distinction is useful:

```python
classifier = pipeline("sentiment-analysis")
```

Here, `classifier` is the **pipeline object**.

Inside that pipeline is a **pretrained classification model**.

```text
pipeline
   │
   ├── tokenizer
   ├── pretrained model
   ├── preprocessing
   └── postprocessing
```

That’s why you can call it like a normal Python function:

```python
classifier("I love this!")
```

instead of manually handling tensors and model outputs.

#### Why DistilBERT here, not Qwen like the API scripts?

Your API scripts used:

```text
Qwen/Qwen2.5-7B-Instruct
```

That is a **chat / instruct LLM** (generates text / answers).

This notebook cell uses:

```text
distilbert/distilbert-base-uncased-finetuned-sst-2-english
```

That is a **sentiment classifier** (outputs POSITIVE / NEGATIVE labels).

| | DistilBERT (here) | Qwen 7B Instruct (your API tests) |
|--|-------------------|-------------------------------------|
| Job | Classification labels | Chat / generate sentences |
| Fits `pipeline("sentiment-analysis")` | Yes | No (wrong model type for this task) |
| Size | Small (~hundreds of MB) | Much larger (~7B) — heavier on Colab |
| How you usually call it | Local pipeline | Remote API / chat style |

```text
sentiment-analysis pipeline  →  needs a classifier model
Qwen instruct                →  needs generation / chat style
```

Same Hugging Face Hub — **different model families for different tasks.**

You *could* ask Qwen “is this positive or negative?” as a chat prompt, but that is **not** what `pipeline("sentiment-analysis")` is built for.

## Example 3 — Under the hood (tokenizer + model)

```text
text → tokenizer → token ids → model → scores → label
```

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
text = "I love learning with Hugging Face pipelines!"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

inputs = tokenizer(text, return_tensors="pt")
print("Token ids (numbers the model sees):")
print(inputs["input_ids"])

with torch.no_grad():
    outputs = model(**inputs)

predicted_id = int(torch.argmax(outputs.logits, dim=-1))
label = model.config.id2label[predicted_id]
print("\nPredicted label:", label)
print("(Same job as pipeline — more code, more control)")

### Explanation — manual path

| Piece | Meaning |
|-------|---------|
| `AutoTokenizer...` | Load tokenizer |
| `AutoModelForSequenceClassification...` | Load model weights |
| `tokenizer(text, ...)` | Text → numbers |
| `model(**inputs)` | Numbers → scores |
| `argmax` / `id2label` | Scores → POSITIVE/NEGATIVE |

**Pipeline does all of this for you.** This cell only shows what was hidden.

## Example 4 — Summarization (another task string)

In [ ]:
summarizer = pipeline("summarization")

long_text = """
Hugging Face pipelines are a high-level API in the transformers library.
They wrap a tokenizer and a model so beginners can run inference quickly.
You choose a task name like sentiment-analysis or summarization.
The pipeline downloads what it needs, runs the model, and returns readable results.
This is different from calling a remote chat API with an OpenAI-compatible client.
"""

summary = summarizer(long_text, max_length=60, min_length=20, do_sample=False)
print(summary)

### Explanation — summarization

| Piece | Meaning |
|-------|---------|
| `pipeline("summarization")` | Same function; different **task string** |
| Long text in | Model compresses meaning |
| Output | Usually includes `summary_text` |

First run may download another model — be patient.

## Example 5 — Question answering with context

Same `pipeline` idea. Task string: `"question-answering"`.

You give a **question** + a **context** passage. The model extracts the answer **from the context** (not free chat like Qwen).

In [ ]:
question = "What are Hugging Face pipelines?"
context = "Pipelines are a high level API for inference of LLMs with common tasks"

device = "cuda" if torch.cuda.is_available() else "cpu"
question_answerer = pipeline("question-answering", device=device)
result = question_answerer(question=question, context=context)
print("device:", device)
print(result)

### Explanation — question answering

```text
question + context  →  pipeline  →  answer span from the context
```

#### The two input strings

| Variable | Meaning |
|----------|---------|
| `question` | What you want to know |
| `context` | Passage the model is allowed to “read” for the answer |

Here the answer is already inside `context` (extractive QA).

#### Build the tool

```python
question_answerer = pipeline("question-answering", device="cuda")
```

| Piece | Meaning |
|-------|---------|
| `"question-answering"` | Task name (**string**), same pattern as sentiment |
| `device=...` | `"cuda"` if GPU available, else `"cpu"` |
| `question_answerer` | **Pipeline object** (like `classifier` / `summarizer`) |

Inside again: tokenizer + pretrained QA model + postprocess.  
`question_answerer` is **not** the model alone — it is the pipeline wrapper.

#### Run it

```python
result = question_answerer(question=question, context=context)
```

Unlike sentiment (one string), QA needs **two** named inputs: `question=` and `context=`.

#### Typical result

```python
{
  'score': 0.95...,
  'start': 14,
  'end': 72,
  'answer': 'a high level API for inference of LLMs with common tasks'
}
```

| Field | Meaning |
|-------|---------|
| `answer` | Text span taken from `context` |
| `score` | Confidence |
| `start` / `end` | Where that span sits in the context string |

#### Vs other pipelines in this notebook

| | Sentiment | Summarization | Question-answering |
|--|-----------|---------------|-------------------|
| Task string | `"sentiment-analysis"` | `"summarization"` | `"question-answering"` |
| Input | one text | one long text | `question` + `context` |
| Output | label | short summary | answer span + score |

```text
pipeline("task-name")  →  tool
tool(...)              →  result
```

#### Important limit

If the answer is **not** in `context`, an extractive QA model may still pick a wrong span.  
This is **not** the same as asking Qwen chat with no passage.

## Example 6 — Image generation (`diffusers`, still Hugging Face Hub)

Same Hub idea as `transformers` pipelines — but a **different library**: `diffusers` (also from Hugging Face).

Model id like `stabilityai/sdxl-turbo` still lives on **huggingface.co**.

In [ ]:
!pip -q install diffusers

from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
)

if torch.cuda.is_available():
    pipe = pipe.to("cuda")

image = pipe(
    prompt="A cute robot reading a book about Hugging Face, digital art",
    num_inference_steps=1,
    guidance_scale=0.0,
).images[0]

display(image)

### Explanation — image generation (still HF Hub)

#### Do we only use Hugging Face models here?

**Yes — Hub models** (hosted on huggingface.co), with Hugging Face libraries.

| This notebook | Library | Example model id | Runs where |
|---------------|---------|------------------|------------|
| Sentiment / QA / summary | `transformers` | DistilBERT, QA defaults, … | Local Colab |
| Your API scripts | HF router / Together | `Qwen/...` | Remote |
| **This example** | **`diffusers`** | `stabilityai/sdxl-turbo` | Local Colab |

`stabilityai/sdxl-turbo` is **Stability AI’s model**, published on the **Hugging Face Hub**. You download it with `from_pretrained(...)` just like transformers models.

#### `transformers.pipeline` vs `diffusers` AutoPipeline

| | `transformers.pipeline` | `diffusers.AutoPipelineForText2Image` |
|--|-------------------------|----------------------------------------|
| Typical use | NLP (sentiment, QA, …) | Images (text → image) |
| Model family | Transformers / LLMs / classifiers | Diffusion models (e.g. SDXL) |
| Same idea? | Task helper wraps model | Helper wraps diffusion pipeline |
| Same Hub? | Yes | Yes |

Course note idea:

> Pipelines can be used for **diffusion models** as well as **transformers**.

Here “pipeline” means the **diffusers** pipeline object (`pipe`), not only `from transformers import pipeline`.

#### Line by line

```python
from diffusers import AutoPipelineForText2Image
```

Import the text-to-image helper from **diffusers** (HF sister library to transformers).

```python
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
)
```

| Piece | Meaning |
|-------|---------|
| `from_pretrained("stabilityai/sdxl-turbo")` | Download / load that Hub model |
| `torch_dtype=torch.float16` | Half precision — less VRAM, faster on GPU |
| `variant="fp16"` | Load the fp16 weight files |
| `pipe` | The diffusion **pipeline object** |

```python
pipe = pipe.to("cuda")
```

Move the pipeline to GPU when available (recommended for SDXL).

```python
image = pipe(prompt="...", num_inference_steps=1, guidance_scale=0.0).images[0]
```

| Piece | Meaning |
|-------|---------|
| `prompt=` | Text description of the image |
| `num_inference_steps=1` | SDXL-Turbo is built for **few steps** (even 1) |
| `guidance_scale=0.0` | Turbo-style setting (follow model card) |
| `.images[0]` | First generated PIL image |

```python
display(image)
```

Show the image in the Colab notebook.

#### Needs

- Colab **GPU** strongly recommended  
- First run downloads a large model (be patient)  
- Optional: `HF_TOKEN` if Hub rate-limits or gated access applies  

#### Tiny story

```text
transformers:  pipeline("sentiment-analysis")  →  text label
diffusers:     AutoPipelineForText2Image(...) →  image
               both pull models from Hugging Face Hub
```

## Recap

1. `pipeline` is a **function** that builds a task tool.
2. The task argument is a **string** (e.g. `"sentiment-analysis"`), not another function.
3. Inside: tokenizer + model + postprocess. The pipeline object is **not** the model alone.
4. Your API scripts = remote. This notebook = local Colab pipeline.
5. Different task strings → different inputs/outputs (sentiment, summarization, question-answering, …).

```text
API:      text → internet → hosted model → text
Pipeline: text → (tokenizer → model → cleanup) on Colab → result
Manual:   you call tokenizer, then model, then interpret
```